# Volume 6, Chapter 2: Metric Spaces and Completeness (Python companion)

Exercises first, solutions at the end. Every exercise is a fixed point found by iteration: the Dottie number, Newton's square roots, PageRank, fractals drawn by contractions or by the chaos game, square roots in the $5$-adic integers, and solutions of differential equations. Requirements: numpy, mpmath, sympy and matplotlib.

*Before each solution, predict what the output will look like.*

## Exercises

**P2.1 (★) The Dottie number.** Iterate $x\mapsto\cos x$ from $x_0=0.1$ and record the errors $|x_n-x^*|$. Estimate the ratio of successive errors and compare it with $\sin x^*$ (Example 2.12). Then run Newton's method on $x-\cos x$ and count correct digits per step.

**P2.2 (★) Newton for $\sqrt2$.** With 60-digit arithmetic, run $x\mapsto\frac12(x+\frac2x)$ from $x_0=2$ and print the number of correct digits of $x_1,\dots,x_6$. Check the claim of Exercise 2.2 that $x_5$ has $24$ correct digits.

**P2.3 (★★) CS application: PageRank is a contraction.** For a directed graph on $n$ pages with transition matrix $P$ (a random surfer follows a uniformly chosen link; pages without links jump uniformly), the Google map is $G(x)=\alpha xP+\frac{1-\alpha}n\mathbf 1$ on probability row vectors, with $\alpha=0.85$. Show that $G$ is an $\alpha$-contraction for $\|\cdot\|_1$, compute PageRank on a random graph of $2000$ pages by iteration, and check the error bound of Theorem 2.11.

**P2.4 (★★) Fractals by iteration and by chance.** Draw the Sierpiński triangle twice: by iterating Hutchinson's map $F$ on a set of points (Theorem 2.19), and by the *chaos game*, which applies one randomly chosen $f_i$ at a time to a single point. Then draw Barnsley's fern from its four affine maps.

**P2.5 (★) $p$-adic square roots.** Compute $\sqrt{-1}\in\Z_5$ to $20$ digits with the contraction $T(x)=x-4(x^2+1)$ of Exercise 2.7, then with Newton's method $x\mapsto x-\frac{x^2+1}{2x}$ in $\Z/5^{20}\Z$. Count the iterations each needs. Also check that $-1=\ldots4444$ in $\Z_5$ and $\frac1{1-5}=\sum5^k$.

**P2.6 (★★) Picard's iteration.** For $y'=y$, $y(0)=1$, compute the Picard iterates symbolically and recognise them (Exercise 2.6). For $y'=\sin y+t$, $y(0)=0$ on $[0,2]$, compute them numerically on a grid and watch $\|y_{n+1}-y_n\|_\infty$ decrease.

## Solutions

### P2.1

In [ ]:
from mpmath import mp, mpf, cos, sin, findroot, log10, nstr
mp.dps = 40
xstar = findroot(lambda x: x - cos(x), 0.7)
print("x* =", nstr(xstar, 30), "  sin x* =", nstr(sin(xstar), 6))
x = mpf("0.1"); errors = []
for n in range(40):
    errors.append(abs(x - xstar)); x = cos(x)
ratios = [errors[n + 1] / errors[n] for n in range(30, 39)]
print("error ratios near the end:", [nstr(r, 6) for r in ratios])

x = mpf("0.1")
for n in range(7):
    print(n, "Newton correct digits:", int(-log10(abs(x - xstar) + mpf(10)**-40)))
    x = x - (x - cos(x)) / (1 + sin(x))

The ratio of successive errors tends to $0.6736\ldots=\sin x^*$, the derivative of $\cos$ at the fixed point in absolute value, as the mean value theorem predicts: $x_{n+1}-x^*=\cos x_n-\cos x^*\approx-\sin x^*(x_n-x^*)$. The minus sign is why the cobweb of Figure 2.2 *spirals*. One more digit costs $\log 10/\log(1/0.6736)\approx5.8$ iterations. Newton's map has derivative $0$ at $x^*$, so once it is close its errors are squared at each step: from $x_0=0.1$ the correct digits go $0,0,2,5,10,22,39$, the last one capped by the $40$-digit precision.

### P2.2

In [ ]:
from mpmath import mp, mpf, sqrt, log10
mp.dps = 60
x, r = mpf(2), sqrt(2)
for n in range(1, 7):
    x = (x + 2 / x) / 2
    e = x - r
    print(n, "error %.3e" % float(e), " correct digits:", int(-log10(e)) if e > 0 else "all")

$x_5$ is off by $9\times10^{-25}$: $24$ correct digits, more than the guaranteed $21$ because the actual first error $e_1\approx0.086$ is smaller than the bound $0.12$, and every later error is roughly its square. Babylonian scribes used this iteration on clay tablets; YBC 7289 gives $\sqrt2$ to six sexagesimal digits.

### P2.3

In [ ]:
import numpy as np
rng = np.random.default_rng(1)
n, alpha = 2000, 0.85
# random web: each page links to a few others, with a preference for popular pages
out_links = []
popularity = rng.pareto(1.5, n) + 1
popularity /= popularity.sum()
for i in range(n):
    k = rng.poisson(5)
    out_links.append(np.unique(rng.choice(n, size=k, p=popularity)) if k else np.array([], dtype=int))

def step(x):
    y = np.zeros(n)
    dangling = 0.0
    for i, links in enumerate(out_links):
        if len(links):
            y[links] += alpha * x[i] / len(links)
        else:
            dangling += alpha * x[i]
    return y + (dangling + 1 - alpha) / n

x = np.full(n, 1 / n)
history = [x]
for _ in range(60):
    x = step(x); history.append(x)
xstar = x
d01 = np.abs(history[1] - history[0]).sum()
for k in [1, 5, 10, 20, 40]:
    err = np.abs(history[k] - xstar).sum()
    print(f"n={k:2d}  ||x_n - x*||_1 = {err:.2e}   a priori bound {alpha**k / (1 - alpha) * d01:.2e}")
top = np.argsort(-xstar)[:5]
print("top pages:", top, "scores", np.round(xstar[top], 4), " sum =", xstar.sum().round(12))

For two probability vectors $x,y$, $\|(x-y)P\|_1\le\|x-y\|_1$ because each row of $P$ is a probability vector (the mass of $x-y$ is redistributed, never created), and the constant term cancels, so $\|G(x)-G(y)\|_1\le\alpha\|x-y\|_1$. The set of probability vectors is closed in $\R^n$, hence complete, and Banach's theorem gives a unique ranking, found by iteration with error at most $\frac{\alpha^n}{1-\alpha}\|x_1-x_0\|_1$. The observed errors sit far below the bound: they shrink by a factor of about $0.45$ per step, so seven digits are gained between $n=20$ and $n=40$. The bound only uses the contraction ratio $\alpha$; the true speed is governed by the second largest eigenvalue of the Google matrix, which is at most $\alpha$ in modulus and here much smaller. Either way convergence is linear, like the cosine's, and choosing $\alpha$ well below $1$ is what makes it fast and the ranking stable.

### P2.4

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
corners = np.array([[0, 0], [1, 0], [0.5, np.sqrt(3) / 2]])
rng = np.random.default_rng(0)

# deterministic: F applied to a cloud of points (start from a random square of points)
pts = rng.random((200, 2))
for _ in range(7):
    pts = np.concatenate([(pts + c) / 2 for c in corners])

# chaos game: one point, one random map at a time
z = np.array([0.3, 0.3]); chaos = []
for k in range(60000):
    z = (z + corners[rng.integers(3)]) / 2
    if k > 20: chaos.append(z)
chaos = np.array(chaos)

# Barnsley's fern
maps = [(np.array([[0, 0], [0, 0.16]]), np.array([0, 0]), 0.01),
        (np.array([[0.85, 0.04], [-0.04, 0.85]]), np.array([0, 1.6]), 0.85),
        (np.array([[0.2, -0.26], [0.23, 0.22]]), np.array([0, 1.6]), 0.07),
        (np.array([[-0.15, 0.28], [0.26, 0.24]]), np.array([0, 0.44]), 0.07)]
probs = [m[2] for m in maps]
z = np.zeros(2); fern = []
for k in range(80000):
    A, b, _ = maps[rng.choice(4, p=probs)]
    z = A @ z + b; fern.append(z)
fern = np.array(fern)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].scatter(*pts.T, s=0.1); ax[0].set_title("iterating F on 200 points (7 steps)")
ax[1].scatter(*chaos.T, s=0.1); ax[1].set_title("chaos game, 60000 points")
ax[2].scatter(*fern.T, s=0.1, c="green"); ax[2].set_title("Barnsley's fern")
for a in ax: a.set_aspect("equal"); a.axis("off")
plt.show()
print("points after 7 deterministic steps:", len(pts))

Both methods draw the same attractor. The deterministic one needs $200\cdot3^7\approx437\,000$ points; the chaos game needs one point and a coin. It works because after $k$ random steps the point is within $2^{-k}$ of the attractor (each map is a $\frac12$-contraction and maps the attractor into itself), and the random choices visit every small piece $f_{i_1}\circ\cdots\circ f_{i_k}(K)$ with frequency $3^{-k}$. Barnsley's fern is the attractor of four affine contractions; the probabilities only change how fast each part fills in, not the set. Barnsley's *collage theorem* (a corollary of Banach's a posteriori bound) turns this around: to encode an image, find a few contractions whose collage is close to it, an idea once used for fractal image compression.

### P2.5

In [ ]:
p, N = 5, 20
mod = p**N
x, it = 2, 0
while (x * x + 1) % mod:
    x = (x - 4 * (x * x + 1)) % mod; it += 1
print("contraction T:", it, "iterations")
digits = lambda x: [(x // p**k) % p for k in range(N)]
print("digits of sqrt(-1):", digits(x))

y, it = 2, 0
while (y * y + 1) % mod:
    y = (y - (y * y + 1) * pow(2 * y, -1, mod)) % mod; it += 1
print("Newton:", it, "iterations; same root:", x == y)
print("-1 =", digits(mod - 1))
print("1/(1-5) =", digits(pow(1 - 5, -1, mod)))

The frozen-derivative map gains one $5$-adic digit per step ($19$ iterations for $20$ digits), Newton doubles them ($5$ iterations), exactly as for $\sqrt2$ in $\R$: Hensel's lemma (Volume 1, Chapter 12) *is* Newton's method in the complete field $\Q_5$. The digits start $2,1,2,1,3,4,\ldots$, matching Exercise 2.7. And $-1=\ldots4444$, $\frac1{1-5}=\ldots1111$, as Theorem 2.10 predicts.

### P2.6

In [ ]:
import sympy as sp
t, s = sp.symbols("t s")
y = sp.Integer(1)
for n in range(5):
    y = 1 + sp.integrate(y.subs(t, s), (s, 0, t))
    print(n + 1, sp.expand(y))

import numpy as np
T, m = 2.0, 4001
grid = np.linspace(0, T, m)
def picard(y):
    integrand = np.sin(y) + grid
    # cumulative trapezoid rule
    return np.concatenate([[0], np.cumsum((integrand[1:] + integrand[:-1]) / 2 * np.diff(grid))])
y = np.zeros(m)
for n in range(1, 13):
    ynew = picard(y)
    print(f"n={n:2d}  sup|y_n - y_(n-1)| = {np.max(np.abs(ynew - y)):.2e}")
    y = ynew
print("y(2) ~", y[-1])

For $y'=y$ the $n$-th iterate is $\sum_{k\le n}\frac{t^k}{k!}$, the Taylor polynomial of $e^t$. For $y'=\sin y+t$ the Lipschitz constant is $L=1$, and the successive differences decrease faster than geometrically, like $\frac{(LT)^n}{n!}$: on a bounded interval Picard's iteration converges even for the plain sup norm. The weighted norm $\|\cdot\|_L$ of Exercise 2.6 is a way to turn this factorial speed into an honest contraction.